#### Imports

#### Installations
py -m pip install -U scikit-learn  
py -m pip install pandas  
py -m pip install networkx  
py -m pip install matplotlib

In [2]:
import json
import math
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
pd.options.display.max_rows = 2000
import subprocess
import networkx as nx
from networkx.algorithms import bipartite
from pathlib import Path  
pd.options.display.max_rows = 400
import matplotlib.pyplot as plt
import locale
import subprocess
import codecs
from xml.etree import ElementTree as ET
locale.setlocale(locale.LC_ALL, 'de-DE.utf-8')

'de-DE.utf-8'

#### Texte

In [3]:
entchen = "Alle meine Entchen schwimmen auf dem See, schwimmen auf dem See, Köpfchen in das Wasser, Schwänzchen in die Höh."
täubchen = "Alle meine Täubchen gurren auf dem Dach, gurren auf dem Dach, fliegt eins in die Lüfte, fliegen alle nach."
texts = [entchen, täubchen]

In [ ]:
seidel = "Wir wußten nicht, wozu wir blühten, Und Jugend schien uns Fluch und Last, Ein Fest an dem wir nicht erglühten, – Man trank – man ging – ein satter Gast. Und unser Blut ging dick und träge, Wir hatten allzu blanke Wehr, Wir hatten allzu glatte Wege, Wir hatten keine Lieder mehr. Drum jauchzen wir in diesen Tagen, Drum sind wir trunken ohne Wein, Drum dröhnt‘s uns aus der Trommeln Schlagen: Oh heil‘ges Glück, heut jung zu sein. "
ball   = "ombula take bitdli solunkola tabla tokta tokta takabia taka tak Babula m‘balam tak tru - ü wo - um biba bimbel o kla o auwa kla o auwa la - auma o kla o ü la o auma klinga - o - e - auwa ome o-auwa klinga inga M ao - Auwa omba dij omuff pomo - auwa tru-ü tro-u-ü o-a-o-ü mo-auwa gomun guma zangaga gago blagaga szagaglugi m ba-o-auma szaga szago szaga la m‘blama bschigi bschigo bschigi bschigi bschiggo bschiggo goggo goggo ogoggo a-o -auma"
trakl  = "Am Abend tönen die herbstlichen Wälder Von tödlichen Waffen, die goldnen Ebenen Und blauen Seen, darüber die Sonne Düstrer hinrollt; umfängt die Nacht Sterbende Krieger, die wilde Klage Ihrer zerbrochenen Münder. Doch stille sammelt im Weidengrund Rotes Gewölk, darin ein zürnender Gott wohnt Das vergoßne Blut sich, mondne Kühle; Alle Straßen münden in schwarze Verwesung. Unter goldnem Gezweig der Nacht und Sternen Es schwankt der Schwester Schatten durch den schweigenden Hain, Zu grüßen die Geister der Helden, die blutenden Häupter; Und leise tönen im Rohr die dunklen Flöten des Herbstes. O stolzere Trauer! ihr ehernen Altäre Die heiße Flamme des Geistes nährt heute ein gewaltiger Schmerz, Die ungebornen Enkel. "
texts  = [seidel, ball, trakl]

#### word count

In [6]:
count_vectorizer = CountVectorizer()
word_count_matrix = count_vectorizer.fit_transform(texts)
feature_names = count_vectorizer.get_feature_names_out()
index = (["Entchen","Täubchen"])
df = pd.DataFrame(word_count_matrix.toarray(), index=index, columns=feature_names)
print (df.transpose())

             Entchen  Täubchen
alle               1         2
auf                2         2
dach               0         2
das                1         0
dem                2         2
die                1         1
eins               0         1
entchen            1         0
fliegen            0         1
fliegt             0         1
gurren             0         2
höh                1         0
in                 2         1
köpfchen           1         0
lüfte              0         1
meine              1         1
nach               0         1
schwimmen          2         0
schwänzchen        1         0
see                2         0
täubchen           0         1
wasser             1         0


#### tf-idf

In [9]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(texts)
feature_names = tfidf_vectorizer.get_feature_names_out()
index = (["Entchen","Täubchen"])
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), index=index, columns=feature_names)
print (tfidf_df.transpose())

              Entchen  Täubchen
alle         0.153115  0.306229
auf          0.306229  0.306229
dach         0.000000  0.430395
das          0.215197  0.000000
dem          0.306229  0.306229
die          0.153115  0.153115
eins         0.000000  0.215197
entchen      0.215197  0.000000
fliegen      0.000000  0.215197
fliegt       0.000000  0.215197
gurren       0.000000  0.430395
höh          0.215197  0.000000
in           0.306229  0.153115
köpfchen     0.215197  0.000000
lüfte        0.000000  0.215197
meine        0.153115  0.153115
nach         0.000000  0.215197
schwimmen    0.430395  0.000000
schwänzchen  0.215197  0.000000
see          0.430395  0.000000
täubchen     0.000000  0.215197
wasser       0.215197  0.000000


#### Reshape dataframe

In [11]:
# Create a Pandas DataFrame from tfidf_matrix and reshape it
reshaped_df = tfidf_df.stack().reset_index()
reshaped_df = reshaped_df.rename(columns={0:'tfidf', 'level_0': 'article','level_1': 'feature'})  
print (reshaped_df)

     article      feature     tfidf
0    Entchen         alle  0.153115
1    Entchen          auf  0.306229
2    Entchen         dach  0.000000
3    Entchen          das  0.215197
4    Entchen          dem  0.306229
5    Entchen          die  0.153115
6    Entchen         eins  0.000000
7    Entchen      entchen  0.215197
8    Entchen      fliegen  0.000000
9    Entchen       fliegt  0.000000
10   Entchen       gurren  0.000000
11   Entchen          höh  0.215197
12   Entchen           in  0.306229
13   Entchen     köpfchen  0.215197
14   Entchen        lüfte  0.000000
15   Entchen        meine  0.153115
16   Entchen         nach  0.000000
17   Entchen    schwimmen  0.430395
18   Entchen  schwänzchen  0.215197
19   Entchen          see  0.430395
20   Entchen     täubchen  0.000000
21   Entchen       wasser  0.215197
22  Täubchen         alle  0.306229
23  Täubchen          auf  0.306229
24  Täubchen         dach  0.430395
25  Täubchen          das  0.000000
26  Täubchen          dem  0

#### Set threshold for tf-idf values

#### generate ToC

In [ ]:
inputfile = "C:/Users/nlutt/myWebsites/KochbuchHD/src/davidis_kochbuch_1849.TEI-P5.xml"
ns = {'tei': 'http://www.tei-c.org/ns/1.0'}
root = ET.parse(open(inputfile, 'r', encoding='utf-8')).getroot()
headings = root.findall(".//tei:body//tei:div[@n='2']/tei:head", ns)
for heading in headings: 
    print (''.join(heading.itertext()))

In [ ]:
a = str(1)
print (a)
a = "Mickey Mouse said \"Hello!\"."
print (a)
a = 'Mickey Mouse said "Hello!".'
print (a)

#### Funktion für die Erzeugung der dot-Datei:

In [ ]:
def graphToDot(graph=None, outfile=None):
   if graph == None:
      print ('Missing graph specification.')
      return
   if outfile == None:
      print ('Missing output file specification.')
      return
   dot = 'graph {\ngraph[rankdir="LR", outputorder="edgesfirst"]\nnode[fontname="Arial", fontsize=120, shape=circle, style=filled, fixedsize=shape];\n'
   for u,v,att in graph.edges(data=True):
      x = [u,v]
      x.sort(key=locale.strxfrm)
      u = x[0]
      v = x[1]
      dot += u+' -- '+v+' [penwidth='+str(att.get('weight'))
      dot += ', id='+'"'+u+"--"+v+'"'
      if att.get('weight') > 1:
            dot += ', color=Red]\n'
      else:
            dot += ']\n'
   for u,att in graph.nodes(data=True):
      dot += u+' [width=' + str(1+3*math.sqrt(att.get('occ'))) +  ']\n'
   dot += '}'
   with codecs.open(outfile, 'w', encoding = 'utf8') as file:
      file.write(dot)
   return


.
└── DGNet_meth_2025
    ├── <other dirs>
    ├── notebooks
    │   ├── .venv
    │   ├── sample_1.ipynb
    │   └── sample_2.ipynb
    └── <other dirs>

#### Funktion für die Berechnung der Keywords und die Erstellung des Graphen:

In [ ]:
def graph_maker(texts, threshold, dot_file, svg_file):

   # Read stopwords
   file = 'C:/Users/nlutt/myOrpheana/03 tools/stopw_german_sorted'
   with open(file, 'r', encoding='utf-8') as stop:
      stopw = stop.read()
   stopw = stopw.split('\n')

   # Fit and transform the article texts using TF-IDF
   vectorizer = TfidfVectorizer(stop_words=stopw)
   tfidf_matrix = vectorizer.fit_transform(texts)
   
   # Create a Pandas DataFrame from tfidf_matrix and reshape it
   text_titles = [f"article_{str(i)}" for i in range(len(texts))]
   feature_names = vectorizer.get_feature_names_out()
   tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), index=text_titles, columns=feature_names)
   tfidf_df = tfidf_df.stack().reset_index()
   tfidf_df = tfidf_df.rename(columns={0:'tfidf', 'level_0': 'article','level_1': 'feature'})  

   # Set threshold for TF-IDF values and determine remaining features
   thres_tfidf = tfidf_df[tfidf_df['tfidf'] > threshold]
   print ('count rows in reduced tf-idf matrix: ', len(thres_tfidf.index))
   print ('tf-idf matrix: ', thres_tfidf)
   # compute document frequency
   occ = thres_tfidf['feature'].value_counts()
   occ_dict = occ.to_dict()
   print ('feature occurences: ', occ_dict)

   # Make semantic graph
   B = nx.Graph(created_by='fruschtique')
   X = nx.Graph(created_by='fruschtique')
   top    = thres_tfidf['article'].unique()
   bottom = thres_tfidf['feature'].unique()
   e_list = []
   for index, row in thres_tfidf.iterrows():
      e_list.append((row['article'], row['feature']))
   B.add_nodes_from(top, bipartite=0)
   B.add_nodes_from(bottom, bipartite=1)
   B.add_edges_from(e_list)
   X = bipartite.weighted_projected_graph(B, bottom)

   # Add attributes to nodes 
   nx.set_node_attributes(X, occ_dict, 'occ')
   print ('count nodes: ',len(X.nodes))
   #for n in X.nodes(data=True):
   #   print (n)
   print ('count edges: ', len(X.edges))
   #for e in X.edges(data=True):
   # print (e) 

   # Generate initial graph
   pos = nx.spring_layout(X)
   nx.draw(X, pos, with_labels=True)   
   plt.show()

   # Gnerate SVG graph and save it
   
   outfile = dot_file
   graphToDot(X, outfile)
   infile  = outfile
   print ('in ', infile)
   outfile = svg_file
   print ('out', outfile)
   sfdp = "C:/Program Files/Graphviz/bin/sfdp.exe"
   subprocess.run ([f"{sfdp}", f"{infile}", '-o', f"{outfile}", 'Goverlap=prism', '-Tsvg'])

In [ ]:
dot_file = r'C:/Users/nlutt/myPublications/2025 DGNet Methodenschule/poems.dot'
svg_file = r'C:/Users/nlutt/myPublications/2025 DGNet Methodenschule/poems.svg'
threshold = 0.115
graph_maker (texts, threshold, dot_file, svg_file)

#### Articles aus 1996-Holländer-Wagner-Nürnberg.json:

In [ ]:
# Read and parse the JSON file
file = 'C:/Users/nlutt/myOrpheana/06 orphs - working copies/1996-Holländer-Wagner-Nürnberg.json'
with open(file, 'r', encoding='utf-8') as f:
    orph = json.load(f)
# Print the parsed data
#print (orph['articles'][3]['author']) 
texts = [article['text'] for article in orph['articles']]
dot_file = r'C:/Users/nlutt/myOrpheana/10 studies/first/first.dot'
svg_file = r'C:/Users/nlutt/myOrpheana/10 studies/first/first.svg'
threshold = 0.12
graph_maker (texts, threshold, dot_file, svg_file)

#### Berechnung der Kosinus-Ähnlichkeit

In [ ]:
def cosine_sim (texts,index):
  # Read stopwords
  file = 'C:/Users/nlutt/Documents/german_sorted'
  with open(file, 'r', encoding='utf-8') as stop:
    stopw = stop.read()
  stopw = stopw.split('\n')
  count_vectorizer = CountVectorizer(stop_words=stopw)
  matrix = count_vectorizer.fit_transform(texts)
  feature_names = count_vectorizer.get_feature_names_out()
  df = pd.DataFrame(matrix.toarray(), index=index, columns=feature_names)
  cs = cosine_similarity(df)
  cs_df = pd.DataFrame(cs, index=index, columns=index)
  print(cs_df)

#### Ähnlichkeit von zwei Artikeln

In [ ]:
# Read and parse the corpus file
file = 'C:/Users/nlutt/myOrpheana/10 studies/second/second_corpus.json'
with open(file, 'r', encoding='utf-8') as f:
    corpus = json.load(f)

In [ ]:
texts = [article['text'] for article in corpus['articles']]
index = [article['author'] for article in corpus['articles']]
cosine_sim (texts,index)